In [ ]:
import pandas as pd
from neo4j import GraphDatabase
import time


driver = GraphDatabase.driver("neo4j://127.0.0.1:7687", auth=("neo4j", "sakuni200211"))


subjects = ['Sugar', 'Gluten', 'Vitamin C', 'Sodium', 'Fiber'] * 1000
predicates = ['increases_risk', 'causes_allergy', 'provides_nutrient', 'linked_to', 'reduces_risk'] * 1000
objects = ['Obesity', 'Celiac Disease', 'Immune Boost', 'Hypertension', 'Heart Health'] * 1000
df = pd.DataFrame({'Subject': subjects, 'Predicate': predicates, 'Object': objects})
print(f"df defined: Shape {df.shape}")

BATCH_SIZE = 5000
count = 0
start_time = time.time()

def add_triples_batch(tx, triples_batch):
    cypher = """
    UNWIND $triples AS row
    MERGE (s:Entity {name: row.s})
    MERGE (o:Entity {name: row.o})
    MERGE (s)-[r:RELATION {type: row.p}]->(o)
    """
    tx.run(cypher, triples=triples_batch)

with driver.session() as session:
    # Connection
    try:
        test_result = session.run("RETURN 1 AS connected")
        if test_result.single()["connected"] == 1:
            print("Connected to Neo4j!")
        else:
            raise ValueError("Connection test failed.")
    except Exception as e:
        print(f"Connection failed: {e}")
        driver.close()
        exit()

    # Vectorize & batch
    triples_list = df[['Subject', 'Predicate', 'Object']].rename(columns={
        'Subject': 's', 'Predicate': 'p', 'Object': 'o'
    }).dropna().to_dict('records')
    print(f"Processing {len(triples_list)} triples in batches...")

    for i in range(0, len(triples_list), BATCH_SIZE):
        batch = triples_list[i:i + BATCH_SIZE]
        try:
            with session.begin_transaction() as tx:
                add_triples_batch(tx, batch)
                tx.commit()
            count += len(batch)
            if count % 30000 == 0:
                elapsed = time.time() - start_time
                print(f"Loaded {count} triples in {elapsed:.1f}s (~{count/elapsed:.0f}/s)")
        except Exception as e:
            print(f"Batch {i//BATCH_SIZE + 1} failed: {e}")
            break

    
    nodes_result = session.run("MATCH (n) RETURN count(n) AS nodes")
    nodes = nodes_result.single()["nodes"]
    
    rels_result = session.run("MATCH ()-[r]->() RETURN count(r) AS rels")
    rels = rels_result.single()["rels"]
    
    print(f"Verification: {nodes} nodes, {rels} rels")

print(f"Loaded {count} triples to Neo4j in {time.time() - start_time:.1f}s total!")
driver.close()

df defined: Shape (5000, 3)
✅ Connected to Neo4j!
Processing 5000 triples in batches...
Verification: 103069 nodes, 527223 rels
Loaded 5000 triples to Neo4j in 0.5s total!


In [6]:
driver = GraphDatabase.driver("neo4j://127.0.0.1:7687", auth=("neo4j", "sakuni200211"))

In [7]:
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
    print("Cleared old data")

Cleared old data


In [8]:
CSV_PATH ="D:/Downloads/Research Project/Data/processed/kg/triples_part_1.csv"

In [9]:
BATCH_SIZE = 10000

In [10]:
def bulk_load(tx, has_name_batch, has_brand_batch, contains_ing_batch):
    """UNWIND bulk MERGE (fast for low power)."""
    if not has_name_batch.empty:
        tx.run("""
            UNWIND $data AS row
            MERGE (f:Food {id: row.sub})
            ON CREATE SET f.name = row.obj
        """, data=[{'sub': r['Subject'], 'obj': r['Object']} for _, r in has_name_batch.iterrows()])
    if not has_brand_batch.empty:
        tx.run("""
            UNWIND $data AS row
            MERGE (f:Food {id: row.sub})
            MERGE (b:Brand {name: row.obj})
            MERGE (f)-[:HAS_BRAND]->(b)
        """, data=[{'sub': r['Subject'], 'obj': r['Object']} for _, r in has_brand_batch.iterrows()])
    if not contains_ing_batch.empty:
        tx.run("""
            UNWIND $data AS row
            MERGE (f:Food {id: row.sub})
            MERGE (i:Ingredient {name: row.obj})
            MERGE (f)-[:CONTAINS_INGREDIENT]->(i)
        """, data=[{'sub': r['Subject'], 'obj': r['Object']} for _, r in contains_ing_batch.iterrows()])

In [ ]:
# Load & prep
print("Loading CSV...")
df = pd.read_csv(CSV_PATH)
df = df.sample(frac=0.1).reset_index(drop=True)  # Downsample
print(f"Using {len(df)} triples (sampled)")

Loading CSV...
Using 500000 triples (sampled)


In [12]:
df_by_pred = {
    'has_name': df[df['Predicate'] == 'has_name'],
    'has_brand': df[df['Predicate'] == 'has_brand'],
    'contains_ingredient': df[df['Predicate'] == 'contains_ingredient']
}
print("Pred counts:", {k: len(v) for k, v in df_by_pred.items()})

Pred counts: {'has_name': 47714, 'has_brand': 47670, 'contains_ingredient': 404616}


In [ ]:
start_time = time.time()
with driver.session() as session:
    # Clear old data
    session.run("MATCH (n) DETACH DELETE n")
    print("Cleared graph")
    
    # Create indexes
    session.run("CREATE INDEX IF NOT EXISTS FOR (f:Food) ON (f.id)")
    session.run("CREATE INDEX IF NOT EXISTS FOR (b:Brand) ON (b.name)")
    session.run("CREATE INDEX IF NOT EXISTS FOR (i:Ingredient) ON (i.name)")
    print("Indexes created—wait 2 min if first run")

    # Batch load per pred
    for pred, df_pred in df_by_pred.items():
        print(f"\n--- Loading {pred} ({len(df_pred)} rows) ---")
        for i in range(0, len(df_pred), BATCH_SIZE):
            batch = df_pred.iloc[i:i + BATCH_SIZE]
            batch_dicts = {
                'has_name_batch': pd.DataFrame() if pred != 'has_name' else batch,
                'has_brand_batch': pd.DataFrame() if pred != 'has_brand' else batch,
                'contains_ing_batch': pd.DataFrame() if pred != 'contains_ingredient' else batch
            }
            session.execute_write(bulk_load, **batch_dicts)
            if (i // BATCH_SIZE + 1) % 5 == 0:  # Progress every 50K
                elapsed = time.time() - start_time
                print(f"  Batch {i // BATCH_SIZE + 1}/{(len(df_pred)-1)//BATCH_SIZE + 1} ({i/len(df_pred)*100:.1f}%) - {elapsed/60:.1f} min total")
        print(f"{pred} loaded!")

Cleared graph
Indexes created—wait 2 min if first run

--- Loading has_name (47714 rows) ---
  Batch 5/5 (83.8%) - 0.3 min total
has_name loaded!

--- Loading has_brand (47670 rows) ---
  Batch 5/5 (83.9%) - 0.4 min total
has_brand loaded!

--- Loading contains_ingredient (404616 rows) ---
  Batch 5/41 (9.9%) - 0.6 min total
  Batch 10/41 (22.2%) - 0.7 min total
  Batch 15/41 (34.6%) - 0.8 min total
  Batch 20/41 (47.0%) - 1.0 min total
  Batch 25/41 (59.3%) - 1.1 min total
  Batch 30/41 (71.7%) - 1.2 min total
  Batch 35/41 (84.0%) - 1.4 min total
  Batch 40/41 (96.4%) - 1.5 min total
contains_ingredient loaded!


In [14]:
total_time = time.time() - start_time
print(f"\nLoad done in {total_time/60:.1f} min! Verify in Browser: MATCH (f:Food)-[:CONTAINS_INGREDIENT]->(i) RETURN count(*) AS rels")

driver.close()


Load done in 2.6 min! Verify in Browser: MATCH (f:Food)-[:CONTAINS_INGREDIENT]->(i) RETURN count(*) AS rels
